# InsectSet66 — Our experiments

Comparative study (same structure as course reference project):

1. **CNN** on Log-Mel spectrogram features
2. **CNN-LSTM** on MFCC features
3. **Traditional classifiers** on hand-crafted statistical features (KNN, Decision Tree, Random Forest, AdaBoost, MLP)

Export figures to `latex/images/` and update `latex/figs/table_results.tex`.

## 1. Imports, config & paths

In [6]:
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

# paths (same as baseline)
ROOT = Path(".").resolve()
MANIFEST_PATH = ROOT / "data/cache/baseline_mel_chunks/manifest.csv"
RESULTS_DIR = ROOT / "results/insect_classification"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# MFCC settings
SAMPLE_RATE = 44100
N_MFCC = 20          # number of MFCC coefficients (13–40 typical)
HOP_LENGTH = 512     # frame step
N_FFT = 2048         # FFT size

print("Manifest:", MANIFEST_PATH)

Manifest: /Users/ali/Desktop/thesis/audio/data/cache/baseline_mel_chunks/manifest.csv


## 2. Load manifest (same 64k chunks as baseline)

In [7]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Loaded {len(manifest)} chunks")
print(manifest["subset"].value_counts())
manifest.head(3)

Loaded 64290 chunks
subset
train         41312
validation    12587
test          10391
Name: count, dtype: int64


,path_name,classID,species_name,subset
0,/Users/ali/Desktop/thesis/audio/data/cache/bas...,60,Roeselianaroeselii,train
1,/Users/ali/Desktop/thesis/audio/data/cache/bas...,60,Roeselianaroeselii,train
2,/Users/ali/Desktop/thesis/audio/data/cache/bas...,60,Roeselianaroeselii,train


## 3. MFCC feature extraction

In [8]:
def extract_mfcc_stats(path: str) -> np.ndarray:
    """Load one 5s chunk and return a fixed-length MFCC stats vector."""
    y, _ = sf.read(path, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE,
                                  n_mfcc=N_MFCC,
                                  n_fft=N_FFT,
                                  hop_length=HOP_LENGTH)
    delta1 = librosa.feature.delta(mfcc, order=1)
    delta2 = librosa.feature.delta(mfcc, order=2)
    # mean + std per coefficient for mfcc, delta1, delta2 → 6 × 20 = 120 features
    features = np.concatenate([
        mfcc.mean(axis=1),   mfcc.std(axis=1),
        delta1.mean(axis=1), delta1.std(axis=1),
        delta2.mean(axis=1), delta2.std(axis=1),
    ])
    return features

In [9]:
# Warm up numba JIT on one file before the full loop (avoids silent 30-min hang)
print("Warming up numba JIT...")
_warm = extract_mfcc_stats(manifest.iloc[0]["path_name"])
print(f"JIT ready. Feature shape: {_warm.shape}")

Warming up numba JIT...
JIT ready. Feature shape: (120,)


In [11]:
from joblib import Parallel, delayed

FEATURE_CACHE = RESULTS_DIR / "mfcc_features.npz"

if FEATURE_CACHE.exists():
    data = np.load(FEATURE_CACHE)
    X, y_labels, subsets = data["X"], data["y"], data["subsets"]
    print("Loaded from cache:", X.shape)
else:
    paths   = manifest["path_name"].tolist()
    classes = manifest["classID"].tolist()
    subs    = manifest["subset"].tolist()

    print(f"Extracting MFCC for {len(paths)} chunks (threaded)...")
    feats_list = Parallel(n_jobs=-1, prefer="threads")(
        delayed(extract_mfcc_stats)(p) for p in tqdm(paths, desc="MFCC")
    )

    X        = np.array(feats_list, dtype=np.float32)
    y_labels = np.array(classes,    dtype=np.int32)
    subsets  = np.array(subs)
    np.savez(FEATURE_CACHE, X=X, y=y_labels, subsets=subsets)
    print("Saved:", X.shape)

Extracting MFCC for 64290 chunks (threaded)...


MFCC:   0%|          | 0/64290 [00:00<?, ?it/s]

Saved: (64290, 120)


## 4. Train / val / test split + normalization

In [12]:
from sklearn.preprocessing import StandardScaler

train_mask = subsets == "train"
val_mask   = subsets == "validation"
test_mask  = subsets == "test"

X_train, y_train = X[train_mask], y_labels[train_mask]
X_val,   y_val   = X[val_mask],   y_labels[val_mask]
X_test,  y_test  = X[test_mask],  y_labels[test_mask]

print(f"Train : {X_train.shape}  labels: {y_train.shape}")
print(f"Val   : {X_val.shape}  labels: {y_val.shape}")
print(f"Test  : {X_test.shape}  labels: {y_test.shape}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
print("Scaled. Train range:", X_train_sc.min().round(2), "→", X_train_sc.max().round(2))

Train : (41312, 120)  labels: (41312,)
Val   : (12587, 120)  labels: (12587,)
Test  : (10391, 120)  labels: (10391,)
Scaled. Train range: -9.69 → 10.0


## 5. Unsupervised analysis — k-means + PCA

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score

IMAGES_DIR = ROOT / "latex" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# --- Load order labels (Orthoptera vs Cicadidae) for coloring ---
species_order_df = pd.read_csv(ROOT / "data" / "66_species_order.csv", header=None, names=["species"])
species_list = species_order_df["species"].tolist()

CICADIDAE = {
    "Aleetacurvicosta","Atrapsaltacollina","Atrapsaltacorticina","Atrapsaltaencaustica",
    "Cicadaorni","Clinopsaltaautumna","Cyclochilaaustralasiae","Diceroproctaeugraphica",
    "Galangalabeculata","Neotibicenpruinosus","Platypleuracfcatenata","Platypleuraplumosa",
    "Platypleurasp10","Platypleurasp12cfhirtipennis","Platypleurasp13","Popplepsaltaaeroides",
    "Popplepsaltanotialis","Psaltodaplaga","Yoyettacelis","Yoyettarepetens",
}
order_labels = ["Cicadidae" if s in CICADIDAE else "Orthoptera" for s in species_list]
id_to_order  = {i: order_labels[i] for i in range(len(species_list))}

train_orders = np.array([id_to_order[c] for c in y_train])

# --- PCA to 2D ---
pca = PCA(n_components=2, random_state=42)
Z_train = pca.fit_transform(X_train_sc)
print(f"PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# --- k-means k=2 (orders) for plot ---
km2 = KMeans(n_clusters=2, random_state=42, n_init=10)
km2_labels = km2.fit_predict(X_train_sc)

# --- k-means k=66 for NMI ---
km66 = KMeans(n_clusters=66, random_state=42, n_init=5, max_iter=200)
km66_labels = km66.fit_predict(X_train_sc)
nmi = normalized_mutual_info_score(y_train, km66_labels)
print(f"NMI (k=66 vs true species): {nmi:.4f}")

# --- Figure: 2x2 panel ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: true order labels
order_colors = {"Orthoptera": "#2196F3", "Cicadidae": "#FF5722"}
for order, color in order_colors.items():
    mask = train_orders == order
    axes[0].scatter(Z_train[mask, 0], Z_train[mask, 1],
                    c=color, label=order, alpha=0.3, s=4, rasterized=True)
axes[0].set_title("PCA — True order labels")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].legend(markerscale=3)

# Right: k-means k=2
cluster_colors = ["#4CAF50", "#9C27B0"]
for k in range(2):
    mask = km2_labels == k
    axes[1].scatter(Z_train[mask, 0], Z_train[mask, 1],
                    c=cluster_colors[k], label=f"Cluster {k}", alpha=0.3, s=4, rasterized=True)
axes[1].set_title(f"PCA — k-means (k=2)\nNMI (k=66): {nmi:.3f}")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].legend(markerscale=3)

plt.suptitle("MFCC feature space — PCA projection (train set)", fontsize=13)
plt.tight_layout()
fig.savefig(IMAGES_DIR / "pca_kmeans_mfcc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: latex/images/pca_kmeans_mfcc.png")

PCA variance explained: 31.1%
NMI (k=66 vs true species): 0.5804
Saved: latex/images/pca_kmeans_mfcc.png


/var/folders/vh/90ygnnbj4gv31x32fybxb_900000gn/T/ipykernel_72182/3204811711.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Classification — SVM + Random Forest

In [14]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import json

# ── SVM ──────────────────────────────────────────────────────────────────────
print("Training SVM (RBF kernel) ...")
svm = SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", random_state=42)
svm.fit(X_train_sc, y_train)

svm_val_pred  = svm.predict(X_val_sc)
svm_test_pred = svm.predict(X_test_sc)

svm_val_acc  = accuracy_score(y_val,  svm_val_pred)
svm_test_acc = accuracy_score(y_test, svm_test_pred)
svm_val_f1   = f1_score(y_val,  svm_val_pred,  average="macro", zero_division=0)
svm_test_f1  = f1_score(y_test, svm_test_pred, average="macro", zero_division=0)

print(f"SVM  val  — acc: {svm_val_acc:.4f}  macro-F1: {svm_val_f1:.4f}")
print(f"SVM  test — acc: {svm_test_acc:.4f}  macro-F1: {svm_test_f1:.4f}")

# ── Random Forest ─────────────────────────────────────────────────────────────
print("\nTraining Random Forest (200 trees) ...")
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_val_pred  = rf.predict(X_val)
rf_test_pred = rf.predict(X_test)

rf_val_acc  = accuracy_score(y_val,  rf_val_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)
rf_val_f1   = f1_score(y_val,  rf_val_pred,  average="macro", zero_division=0)
rf_test_f1  = f1_score(y_test, rf_test_pred, average="macro", zero_division=0)

print(f"RF   val  — acc: {rf_val_acc:.4f}  macro-F1: {rf_val_f1:.4f}")
print(f"RF   test — acc: {rf_test_acc:.4f}  macro-F1: {rf_test_f1:.4f}")

# ── Save metrics ──────────────────────────────────────────────────────────────
metrics = {
    "SVM":  {"val_acc": svm_val_acc,  "test_acc": svm_test_acc,
             "val_f1":  svm_val_f1,   "test_f1":  svm_test_f1},
    "RF":   {"val_acc": rf_val_acc,   "test_acc": rf_test_acc,
             "val_f1":  rf_val_f1,    "test_f1":  rf_test_f1},
}
with open(RESULTS_DIR / "mfcc_classifier_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nMetrics saved.")

Training SVM (RBF kernel) ...
SVM  val  — acc: 0.7445  macro-F1: 0.6327
SVM  test — acc: 0.7671  macro-F1: 0.7112

Training Random Forest (200 trees) ...
RF   val  — acc: 0.7481  macro-F1: 0.6006
RF   test — acc: 0.7541  macro-F1: 0.6373

Metrics saved.


In [15]:
def plot_confusion(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, ax=ax, cmap="Blues", xticklabels=False, yticklabels=False)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_confusion(y_test, svm_test_pred,
               f"SVM confusion matrix (test) — acc {svm_test_acc:.3f}  macro-F1 {svm_test_f1:.3f}",
               IMAGES_DIR / "confusion_matrix_svm.png")

plot_confusion(y_test, rf_test_pred,
               f"RF confusion matrix (test)  — acc {rf_test_acc:.3f}  macro-F1 {rf_test_f1:.3f}",
               IMAGES_DIR / "confusion_matrix_rf.png")

Saved: /Users/ali/Desktop/thesis/audio/latex/images/confusion_matrix_svm.png
Saved: /Users/ali/Desktop/thesis/audio/latex/images/confusion_matrix_rf.png


/var/folders/vh/90ygnnbj4gv31x32fybxb_900000gn/T/ipykernel_72182/1038859752.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/vh/90ygnnbj4gv31x32fybxb_900000gn/T/ipykernel_72182/1038859752.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Model interpretability — RF feature importance

In [16]:
# Build feature names: mfcc_mean_0 ... delta2_std_19
feat_names = (
    [f"mfcc_mean_{i}"   for i in range(N_MFCC)] +
    [f"mfcc_std_{i}"    for i in range(N_MFCC)] +
    [f"delta1_mean_{i}" for i in range(N_MFCC)] +
    [f"delta1_std_{i}"  for i in range(N_MFCC)] +
    [f"delta2_mean_{i}" for i in range(N_MFCC)] +
    [f"delta2_std_{i}"  for i in range(N_MFCC)]
)

importances = rf.feature_importances_
top_k = 20
top_idx = np.argsort(importances)[::-1][:top_k]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_k), importances[top_idx][::-1], color="#2196F3")
ax.set_yticks(range(top_k))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=9)
ax.set_xlabel("Gini importance")
ax.set_title(f"Top {top_k} MFCC features — Random Forest (200 trees)")
plt.tight_layout()
fig.savefig(IMAGES_DIR / "rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: latex/images/rf_feature_importance.png")

Saved: latex/images/rf_feature_importance.png


/var/folders/vh/90ygnnbj4gv31x32fybxb_900000gn/T/ipykernel_72182/1389025698.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Results summary — comparison with mel-CNN baseline

In [17]:
results_table = pd.DataFrame([
    {"Model": "SVM (MFCC)",          "Features": "MFCC stats",  "Test Acc": f"{svm_test_acc:.4f}", "Macro-F1": f"{svm_test_f1:.4f}"},
    {"Model": "Random Forest (MFCC)","Features": "MFCC stats",  "Test Acc": f"{rf_test_acc:.4f}",  "Macro-F1": f"{rf_test_f1:.4f}"},
    {"Model": "Mel-CNN (replicated)","Features": "Log-mel",     "Test Acc": "0.6037",              "Macro-F1": "0.5144"},
    {"Model": "Mel-CNN (Faiss et al.)","Features": "Log-mel",   "Test Acc": "~0.82",               "Macro-F1": "—"},
])
print(results_table.to_string(index=False))

# Save for Overleaf
results_table.to_csv(RESULTS_DIR / "results_summary.csv", index=False)
print("\nSaved: results/insect_classification/results_summary.csv")

                 Model   Features Test Acc Macro-F1
            SVM (MFCC) MFCC stats   0.7671   0.7112
  Random Forest (MFCC) MFCC stats   0.7541   0.6373
  Mel-CNN (replicated)    Log-mel   0.6037   0.5144
Mel-CNN (Faiss et al.)    Log-mel    ~0.82        —

Saved: results/insect_classification/results_summary.csv
